## Objectives

This notebook develops forecasting models to predict future product demand.

The following models will be implemented:

- Moving Average Forecast
- ARIMA Forecast

The models will be evaluated using:

- RMSE
- MAPE

Finally, a 90-day demand forecast will be generated.

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt

plt.style.use("ggplot")

In [ ]:
df = pd.read_csv(
    "data/processed/clean_sales.csv",
    parse_dates=["date"]
)

df.head()

In [ ]:
df.set_index("date", inplace=True)

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(
    df.index,
    df["sales"]
)

plt.title("Historical Daily Sales")

plt.xlabel("Date")

plt.ylabel("Sales")

plt.show()

In [ ]:
train_size = int(len(df) * 0.8)

train = df.iloc[:train_size]

test = df.iloc[train_size:]

In [ ]:
print("Training Records:", len(train))
print("Testing Records :", len(test))

In [ ]:
import matplotlib.pyplot as plt

plt.close("all")

plt.figure(figsize=(15,5))

# Downsample for plotting only
train_plot = train.iloc[::100]
test_plot = test.iloc[::100]

plt.plot(
    train_plot.index,
    train_plot["sales"],
    label="Train"
)

plt.plot(
    test_plot.index,
    test_plot["sales"],
    label="Test"
)

plt.legend(loc="upper left")

plt.title("Train/Test Split")

plt.show()

In [ ]:
from src.forecasting import (
    moving_average_forecast,
    train_arima,
    arima_forecast,
    forecast_future
)

In [ ]:
import gc

# Delete variables you no longer need
# del some_large_dataframe
# del some_large_array

gc.collect()

In [ ]:
ma_predictions = moving_average_forecast(
    train,
    test,
    column="sales",
    window=7
)

print(ma_predictions[:10])

In [ ]:
plt.figure(figsize=(15,5))

plt.plot(
    test.index,
    test["sales"],
    label="Actual Sales",
    linewidth=2
)

plt.plot(
    test.index,
    ma_predictions,
    label="Moving Average Forecast",
    linestyle="--"
)

plt.title("Moving Average Forecast vs Actual Sales")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.legend()

plt.show()

In [ ]:
print(train.index)
print(type(train.index))

In [ ]:
import pandas as pd

df = pd.read_csv("data/processed/clean_sales.csv")

In [ ]:
train = df[df["date"] < "2023-01-01"]
test = df[df["date"] >= "2023-01-01"]

In [ ]:
print(train.columns)
print(train.head())

In [ ]:
import pandas as pd

train["date"] = pd.to_datetime(train["date"])

train_daily = (
    train.groupby("date", as_index=True)["sales"]
         .sum()
         .to_frame()
)

train_daily.index = pd.DatetimeIndex(train_daily.index)
train_daily = train_daily.asfreq("D")

print(train_daily.head())
print(train_daily.index.freq)


In [ ]:
arima_model = train_arima(
    train_daily,
    column="sales",
    order=(5,1,0)
)

In [ ]:
test["date"] = pd.to_datetime(test["date"])

test_daily = (
    test.groupby("date")["sales"]
        .sum()
        .to_frame()
)

print(test_daily.shape)

In [ ]:
arima_predictions = arima_forecast(
    arima_model,
    len(test_daily)
)

print(arima_predictions[:10])

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

rmse = np.sqrt(
    mean_squared_error(
        test_daily["sales"],
        arima_predictions
    )
)

mae = mean_absolute_error(
    test_daily["sales"],
    arima_predictions
)

print("RMSE:", rmse)
print("MAE:", mae)

In [ ]:
import matplotlib.pyplot as plt
import gc

gc.collect()

plt.figure(figsize=(15,5))

plt.plot(
    test_daily.index,
    test_daily["sales"],
    label="Actual"
)

plt.plot(
    test_daily.index,
    arima_predictions,
    label="ARIMA Forecast"
)

plt.title("ARIMA Forecast")
plt.xlabel("Date")
plt.ylabel("Sales")

plt.legend()
plt.show()

In [ ]:
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error
)
import numpy as np

In [ ]:
from collections import deque
import numpy as np
from src.forecasting import moving_average_forecast

ma_predictions = moving_average_forecast(
    train_daily,
    test_daily,
    column="sales",
    window=7
)

print(ma_predictions[:10])

In [ ]:
print(len(test_daily))
print(len(ma_predictions))
print(len(arima_predictions))

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

ma_rmse = np.sqrt(
    mean_squared_error(
        test_daily["sales"],
        ma_predictions
    )
)

arima_rmse = np.sqrt(
    mean_squared_error(
        test_daily["sales"],
        arima_predictions
    )
)

ma_mae = mean_absolute_error(
    test_daily["sales"],
    ma_predictions
)

arima_mae = mean_absolute_error(
    test_daily["sales"],
    arima_predictions
)

print("Moving Average RMSE:", ma_rmse)
print("ARIMA RMSE:", arima_rmse)

print("Moving Average MAE:", ma_mae)
print("ARIMA MAE:", arima_mae)

In [ ]:
ma_mape = (
    np.abs(
        (test_daily["sales"] - ma_predictions)
        / test_daily["sales"]
    ).mean()
) * 100

arima_mape = (
    np.abs(
        (test_daily["sales"] - arima_predictions)
        / test_daily["sales"]
    ).mean()
) * 100

In [ ]:
comparison = pd.DataFrame({
    "Model": ["Moving Average", "ARIMA"],
    "RMSE": [ma_rmse, arima_rmse],
    "MAE": [ma_mae, arima_mae],
    "MAPE (%)": [ma_mape, arima_mape]
})

comparison

In [ ]:
import gc

gc.collect()

plt.figure(figsize=(7,5))

plt.bar(
    comparison["Model"],
    comparison["RMSE"]
)

plt.title("RMSE Comparison of Forecasting Models")
plt.ylabel("RMSE")

plt.show()

In [ ]:
best_model = comparison.loc[
    comparison["RMSE"].idxmin(),
    "Model"
]

print(f"Best Performing Model: {best_model}")

In [ ]:
from src.forecasting import forecast_future

future_forecast = forecast_future(
    train_daily,
    column="sales",
    periods=90
)

future_forecast.head()

In [ ]:
import gc
import matplotlib.pyplot as plt

gc.collect()

plt.figure(figsize=(15,5))

plt.plot(
    train_daily.index,
    train_daily["sales"],
    label="Historical Sales"
)

plt.plot(
    future_forecast["date"],
    future_forecast["forecast_sales"],
    color="red",
    label="90-Day Forecast"
)

plt.title("Historical Sales with 90-Day Forecast")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.legend()

plt.show()

In [ ]:
future_forecast.to_csv(
    "data/processed/future_forecast.csv",
    index=False
)

print("Future forecast saved successfully!")

# Conclusion

In this notebook, two forecasting approaches were implemented and evaluated:

- Moving Average
- ARIMA

The models were assessed using RMSE, MAE, and MAPE. Based on the evaluation metrics, the better-performing model was selected to generate a 90-day demand forecast.

The forecast has been saved and will be integrated into the Streamlit dashboard in Week 4.